# 04 — Stratification by TCAB1/WRAP53

**First notebook gated on the frozen pre-specification.** Applies prespec §4 and records
the freeze commit hash.

Per §4d (v1.1), the §4b minimum-cell guard fired: 3 of 16 ALT+ lines fall below the
bottom-tertile cut, below the <4-per-arm threshold. So:

| | Rule | Split |
|---|---|---|
| **Primary** | median within ALT+ | 8 v 8 |
| **Pre-specified sensitivity** | §4a OS-reference tertile (superseded) | 3 v 13 |

Both are constructed here and both are carried into notebook 05 in parallel.

Per §4c the stratifier is **direct WRAP53 log2(TPM+1)** (notebook 02 selected the
`direct` branch).

Reads no gene-effect values.

## 1 — Header and freeze commit

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

from src.utils import git_head

# The freeze commit is a FIXED historical fact, not whatever HEAD happens to be
# when this notebook is re-run. Recording git_head() here labelled every later
# commit as "the freeze commit" and made the report differ between faithful
# reproductions. The run-time HEAD still goes to provenance.jsonl.
FREEZE_COMMIT_V10 = '81745ed'   # prespec v1.0 frozen
FREEZE_COMMIT_V11 = 'a63d56c'   # prespec v1.1, amended before any outcome was read
FREEZE_COMMIT = FREEZE_COMMIT_V10
RUNTIME_HEAD = git_head()
print(f'\nPre-specification freeze commit (v1.0): {FREEZE_COMMIT_V10}')
print(f'Pre-specification amendment (v1.1)   : {FREEZE_COMMIT_V11}')
print(f'git HEAD at this run                 : {RUNTIME_HEAD}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T01:23:18Z
Python: 3.12.13

Pre-specification freeze commit (v1.0): 81745ed
Pre-specification amendment (v1.1)   : a63d56c
git HEAD at this run                 : ff193b5f2849f71b5b7a2e025b730481e1d6a9e0


## 2 — Gate check

In [2]:
from src.cohort import build_cohort, assert_all, gold_only
from src.stratify import (
    load_expression, load_wrap53_cn, osteosarcoma_rna_models, tertile_cut, stratify,
    stratify_median, stratify_tertile_within,
    TCAB1_GENE, TP53_GENE, LOW_LABEL, INTACT_LABEL, MIN_CELL_SIZE, TERTILE_Q,
)
import json as _json
from src.utils import PROCESSED_DIR
leaked = [m for m in sys.modules if 'dependency_stats' in m]
assert not leaked, f'OUTCOME LEAK: {leaked} imported in this notebook'
print('Gate check passed: src.dependency_stats is NOT loaded.')
print('CRISPRGeneEffect.csv is unreachable from this notebook.')

Gate check passed: src.dependency_stats is NOT loaded.
CRISPRGeneEffect.csv is unreachable from this notebook.


## 3 — Confirm the notebook 02 branch

In [3]:
strand = _json.loads((PROCESSED_DIR / 'strand_decision.json').read_text())
BRANCH = strand['branch']
print(f"notebook 02 branch: {BRANCH}")
assert BRANCH in ('direct', 'direct_cn_corroborated'), (
    f"Branch {BRANCH!r} requires the CN-conjunction stratifier; this notebook implements "
    f"the direct stratifier only. See prespec §4c."
)
print('-> stratifier = direct WRAP53 log2(TPM+1), per prespec §4c')

notebook 02 branch: direct
-> stratifier = direct WRAP53 log2(TPM+1), per prespec §4c


## 4 — Load the exposure

In [4]:
cohort = build_cohort()
assert_all(cohort)
alt_ids = cohort.alt_positive['ModelID'].tolist()
neg_ids = cohort.alt_negative['ModelID'].tolist()
names = pd.concat([cohort.alt_positive, cohort.alt_negative]).set_index('ModelID')['StrippedCellLineName']

os_ids = osteosarcoma_rna_models()
expr = load_expression(sorted(set(os_ids) | set(alt_ids) | set(neg_ids)), [TCAB1_GENE, TP53_GENE])
cn = load_wrap53_cn(sorted(set(os_ids) | set(alt_ids) | set(neg_ids)))
wrap53 = expr[TCAB1_GENE]

print(f'reference set (osteosarcoma lines with RNA): {len(os_ids)}')
print(f'ALT+ lines: {len(alt_ids)}   ALT- lines: {len(neg_ids)}')
print(f'WRAP53 CN available for {cn.notna().sum()} of {len(cn)} lines')

reference set (osteosarcoma lines with RNA): 34
ALT+ lines: 16   ALT- lines: 9
WRAP53 CN available for 33 of 34 lines


## 5 — §4a tertile rule and the §4b guard

In [5]:
cut = tertile_cut(wrap53.loc[os_ids])
n_below = int((wrap53.loc[alt_ids] < cut).sum())
print(f'§4a tertile cut ({TERTILE_Q:.4f} quantile over {len(os_ids)} OS lines) = {cut:.6f}')
print(f'ALT+ lines below the cut: {n_below} of {len(alt_ids)}')
print(f'§4b guard threshold: <{MIN_CELL_SIZE} per arm')
print()

primary = stratify(alt_ids, reference_ids=os_ids, expression=wrap53)
for note in primary.notes:
    print(f'  {note}')
print()
print(f'PRIMARY rule   : {primary.rule}')
print(f'PRIMARY sizes  : {primary.sizes}')
assert primary.guard_triggered, (
    'Prespec §4d records that the guard fired. It did not fire now, so the WRAP53 '
    'distribution or the cohort has changed -- investigate before proceeding.'
)
print('\n-> matches prespec §4d: guard fired, primary is the 8v8 median split')

§4a tertile cut (0.3333 quantile over 34 OS lines) = 3.335580
ALT+ lines below the cut: 3 of 16
§4b guard threshold: <4 per arm

  Cutpoint = 0.3333 quantile of WRAP53 log2(TPM+1) over 34 osteosarcoma lines with RNA = 3.335580
  Applied to 16 ALT+ lines -> 3 TCAB1-low / 13 TCAB1-intact
  TCAB1-low arm n=3 (intact n=13); minimum cell size is 4. Tertile primary declared NOT ESTIMABLE; falling back to the median split within the ALT+ set, per the pre-specified Section 4 guard.
  Median cutpoint within ALT+ = 3.794467

PRIMARY rule   : median_within_alt_pos
PRIMARY sizes  : {'TCAB1-low': 8, 'TCAB1-intact': 8}

-> matches prespec §4d: guard fired, primary is the 8v8 median split


## 6 — The two contrasts carried into notebook 05

In [6]:
strata_primary = primary.strata                                   # 8v8 median
strata_tertile = pd.Series(
    np.where(wrap53.loc[alt_ids] < cut, LOW_LABEL, INTACT_LABEL),
    index=wrap53.loc[alt_ids].index, name='tcab1_stratum_tertile',
)   # 3v13 OS-reference tertile (the superseded §4a primary)

CONTRASTS = {
    'primary_median_8v8': strata_primary,
    'sensitivity_tertile_3v13': strata_tertile,
}
for label, s in CONTRASTS.items():
    counts = s.value_counts().to_dict()
    print(f'{label:<28s} {counts}')

# ALT- negative control, same median rule applied within the ALT- set
strata_alt_negative = stratify_median(neg_ids, wrap53)
print(f'{"alt_negative_median":<28s} {strata_alt_negative.value_counts().to_dict()}')

primary_median_8v8           {'TCAB1-intact': 8, 'TCAB1-low': 8}
sensitivity_tertile_3v13     {'TCAB1-intact': 13, 'TCAB1-low': 3}
alt_negative_median          {'TCAB1-intact': 5, 'TCAB1-low': 4}


## 7 — §4d assertion: the two WRAP53-null lines must be in both low arms

Prespec §4d requires ACH-002834 (OS384) and ACH-003185 (NRHOS4) to appear in the
TCAB1-low arm of **both** contrasts, and to flag immediately if either drops out.

In [7]:
WRAP53_NULL_LINES = {'ACH-002834': 'OS384', 'ACH-003185': 'NRHOS4'}

rows = []
for model_id, name in WRAP53_NULL_LINES.items():
    rows.append({
        'ModelID': model_id, 'cell_line': name,
        'WRAP53_log2tpm': wrap53[model_id],
        'TP53_log2tpm': expr.loc[model_id, TP53_GENE],
        'WRAP53_cn': cn[model_id],
        'alt_status': cohort.alt_positive.set_index('ModelID').loc[model_id, 'alt_status'],
        'arm_primary_8v8': strata_primary[model_id],
        'arm_tertile_3v13': strata_tertile[model_id],
    })
null_lines = pd.DataFrame(rows)
print(null_lines.to_string(index=False))
print()
for _, r in null_lines.iterrows():
    for col in ('arm_primary_8v8', 'arm_tertile_3v13'):
        assert r[col] == LOW_LABEL, (
            f"PRESPEC §4d VIOLATION: {r['cell_line']} ({r['ModelID']}) is in "
            f"{r[col]!r} for {col}, expected {LOW_LABEL!r}. Flag immediately."
        )
print(f'§4d assertion PASSED: both WRAP53-null lines are in the {LOW_LABEL} arm of both contrasts.')

   ModelID cell_line  WRAP53_log2tpm  TP53_log2tpm  WRAP53_cn        alt_status arm_primary_8v8 arm_tertile_3v13
ACH-002834     OS384             0.0      0.000000   0.000090 alt_gold_positive       TCAB1-low        TCAB1-low
ACH-003185    NRHOS4             0.0      0.105981   0.000008        alt_silver       TCAB1-low        TCAB1-low

§4d assertion PASSED: both WRAP53-null lines are in the TCAB1-low arm of both contrasts.


## 8 — Stratifier table and CN concordance

In [8]:
table = pd.DataFrame({
    'cell_line': names.reindex(alt_ids),
    'WRAP53_log2tpm': wrap53.loc[alt_ids],
    'TP53_log2tpm': expr.loc[alt_ids, TP53_GENE],
    'WRAP53_cn': cn.reindex(alt_ids),
    'arm_primary_8v8': strata_primary.reindex(alt_ids),
    'arm_tertile_3v13': strata_tertile.reindex(alt_ids),
    'alt_status': cohort.alt_positive.set_index('ModelID')['alt_status'].reindex(alt_ids),
}).sort_values('WRAP53_log2tpm')
print(table.to_string())
print()
valid = table['WRAP53_cn'].notna()
rho_cn = table.loc[valid, 'WRAP53_log2tpm'].corr(table.loc[valid, 'WRAP53_cn'], method='spearman')
print(f'Spearman(WRAP53 expression, WRAP53 CN) within ALT+ lines: {rho_cn:+.4f}  (n={int(valid.sum())})')

                  cell_line  WRAP53_log2tpm  TP53_log2tpm  WRAP53_cn arm_primary_8v8 arm_tertile_3v13         alt_status
ModelID                                                                                                                 
ACH-002834            OS384        0.000000      0.000000   0.000090       TCAB1-low        TCAB1-low  alt_gold_positive
ACH-003185           NRHOS4        0.000000      0.105981   0.000008       TCAB1-low        TCAB1-low         alt_silver
ACH-003171           NRHOS1        2.748135      2.531331   0.703361       TCAB1-low        TCAB1-low         alt_silver
ACH-002433    CCLFPEDS0019T        3.335580      6.541247   1.311482       TCAB1-low     TCAB1-intact         alt_silver
ACH-003178           NRHOS2        3.426647      1.056812   0.661215       TCAB1-low     TCAB1-intact         alt_silver
ACH-001715            CAL72        3.458638      0.290298   0.686669       TCAB1-low     TCAB1-intact  alt_gold_positive
ACH-003573             C337     

## 9 — Persist and report

In [9]:
strata_path = PROCESSED_DIR / 'tcab1_strata.csv'
out = table.copy()
out.index.name = 'ModelID'
out.to_csv(strata_path)

neg_path = PROCESSED_DIR / 'tcab1_strata_alt_negative.csv'
neg_table = pd.DataFrame({
    'cell_line': names.reindex(neg_ids),
    'WRAP53_log2tpm': wrap53.loc[neg_ids],
    'WRAP53_cn': cn.reindex(neg_ids),
    'arm_median': strata_alt_negative.reindex(neg_ids),
}).sort_values('WRAP53_log2tpm')
neg_table.index.name = 'ModelID'
neg_table.to_csv(neg_path)

lines = [
    '# Notebook 04 — Stratification by TCAB1/WRAP53',
    '',
    f'**Pre-specification freeze commit:** `{FREEZE_COMMIT_V10}` (v1.0), '
    f'amended at `{FREEZE_COMMIT_V11}` (v1.1, before any gene-effect value was read).',
    '',
    f'Stratifier: direct WRAP53 log2(TPM+1) (notebook 02 branch `{BRANCH}`, prespec §4c).',
    'Reads no gene-effect values.',
    '',
    '## §4b minimum-cell guard: FIRED',
    '',
    f'- §4a tertile cut ({TERTILE_Q:.4f} quantile over {len(os_ids)} osteosarcoma lines with RNA) = **{cut:.6f}**',
    f'- ALT+ lines below the cut: **{n_below} of {len(alt_ids)}**',
    f'- Guard threshold: fewer than {MIN_CELL_SIZE} per arm',
    f'- Outcome: tertile primary declared NOT ESTIMABLE; primary falls back to the median split',
    '',
    f'> {primary.guard_reason}',
    '',
    'This matches prespec §4d, which recorded the guard outcome before any gene-effect',
    'value was read. The guard was committed in v1.0 before the WRAP53 distribution was',
    'examined, so the fallback is pre-specified, not post-hoc.',
    '',
    '## Contrasts carried into notebook 05',
    '',
    '| Contrast | Rule | Split | Role |',
    '|---|---|---|---|',
    f'| `primary_median_8v8` | median within ALT+ | {strata_primary.value_counts().get(LOW_LABEL,0)} v {strata_primary.value_counts().get(INTACT_LABEL,0)} | **primary** |',
    f'| `sensitivity_tertile_3v13` | §4a OS-reference tertile (superseded) | {strata_tertile.value_counts().get(LOW_LABEL,0)} v {strata_tertile.value_counts().get(INTACT_LABEL,0)} | pre-specified sensitivity |',
    f'| `alt_negative_median` | median within ALT- | {strata_alt_negative.value_counts().get(LOW_LABEL,0)} v {strata_alt_negative.value_counts().get(INTACT_LABEL,0)} | §6.2 negative control |',
    '',
    'Both ALT+ contrasts are run in parallel in notebook 05 with the same test and the',
    'same output format, and both enter the final report (prespec §4d).',
    '',
    '## §4d assertion: WRAP53-null lines',
    '',
    'Both lines are required to be in the TCAB1-low arm of both contrasts. **PASSED.**',
    '',
    '| ModelID | Cell line | ALT tier | WRAP53 | TP53 | WRAP53 CN | Arm (8v8) | Arm (3v13) |',
    '|---|---|---|---|---|---|---|---|',
]
for _, r in null_lines.iterrows():
    lines.append(
        f"| {r['ModelID']} | {r['cell_line']} | {r['alt_status']} "
        f"| {r['WRAP53_log2tpm']:.3f} | {r['TP53_log2tpm']:.3f} | {r['WRAP53_cn']:.5f} "
        f"| {r['arm_primary_8v8']} | {r['arm_tertile_3v13']} |"
    )
lines += [
    '',
    'Recorded as an exposure-side measurement fact with copy-number corroboration.',
    '**No mechanistic claim is drawn from it here**; interpretation waits for notebooks',
    '05 and 07 (prespec §4d).',
    '',
    '## Copy-number concordance',
    '',
    f'Spearman(WRAP53 expression, WRAP53 CN) within the ALT+ lines: **{rho_cn:+.4f}** '
    f'(n={int(valid.sum())}). Independent corroboration of the `direct` branch: antisense',
    'cross-assignment would degrade this concordance.',
    '',
    '## Stratifier table (ALT+ lines, ascending WRAP53)',
    '',
    '| ModelID | Cell line | WRAP53 | TP53 | WRAP53 CN | Arm (8v8) | Arm (3v13) | ALT tier |',
    '|---|---|---|---|---|---|---|---|',
]
for model_id, r in table.iterrows():
    cn_str = f"{r['WRAP53_cn']:.4f}" if pd.notna(r['WRAP53_cn']) else 'NA'
    lines.append(
        f"| {model_id} | {r['cell_line']} | {r['WRAP53_log2tpm']:.3f} "
        f"| {r['TP53_log2tpm']:.3f} | {cn_str} | {r['arm_primary_8v8']} "
        f"| {r['arm_tertile_3v13']} | {r['alt_status']} |"
    )
report = write_report('04', lines)
manifest = write_sha256_manifest('04', [report, strata_path, neg_path])
record_provenance([{
    'notebook': '04', 'run_timestamp': RUN_TIMESTAMP,
    'freeze_commit_v1_0': FREEZE_COMMIT_V10, 'freeze_commit_v1_1': FREEZE_COMMIT_V11,
    'runtime_git_head': RUNTIME_HEAD,
    'branch': BRANCH, 'guard_triggered': bool(primary.guard_triggered),
    'tertile_cut': float(cut), 'n_below_cut': n_below,
    'primary_rule': primary.rule, 'primary_sizes': primary.sizes,
}])
print(f'wrote {strata_path.name}')
print(f'wrote {neg_path.name}')
print(f'wrote {report.name}')
print(f'wrote {manifest.name}')

wrote tcab1_strata.csv
wrote tcab1_strata_alt_negative.csv
wrote notebook_04_report.md
wrote notebook_04_sha256.json
